# MultiTurnMCPUseMetric

## What it measures

Whether an agent used **MCP primitives** correctly across a multi-turn conversation. For
each unit interaction it produces two sub-scores against the declared MCP servers:

- **primitive accuracy** - were the right MCP tools, resources and prompts used for the task?
- **argument accuracy** - did each invocation match the tool's declared input schema?

The final score is `min(mean(primitive), mean(argument))`, so an agent cannot compensate for
malformed arguments by choosing well, or vice versa.

The difference from `ToolUseMetric` is the frame of reference. `ToolUseMetric` judges
against a flat list of tools you supply. This metric judges against **declared MCP servers**
- each with its transport, its tool list and each tool's own `inputSchema` - so it is
evaluating protocol-level usage, not just tool choice.

## When it is useful

On any agent whose capabilities come from MCP servers, particularly when servers are added,
versioned or swapped. It is the metric that notices an agent calling a tool whose schema has
changed underneath it, or reaching into the wrong server for a capability that exists in
both.

## DeepEval inputs and test-case type

| DeepEval field | Required |
|---|---|
| test case type | `ConversationalTestCase` |
| `turns` | yes - assistant turns carry `mcp_tools_called` |
| `mcp_servers` | yes - `list[MCPServer]`; the metric raises `MissingTestCaseParamsError` if empty |

`Turn.mcp_tools_called` is strictly typed: each entry must be an `MCPToolCall` whose
`result` is an `mcp.types.CallToolResult`. A plain dict is rejected at construction.

In [ ]:
# --------------------------------------------------------------------------
# Configuration. Every value comes from the environment - nothing about this
# machine, this port or this deployment is baked into the notebook.
# --------------------------------------------------------------------------
import json
import os
import textwrap
from pathlib import Path

import httpx
from dotenv import load_dotenv

# Look for .env next to the notebook, then one level up (the project root).
for _candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if _candidate.is_file():
        load_dotenv(_candidate)
        break


class MissingConfiguration(RuntimeError):
    """Raised when a required environment variable is absent."""


def env(name, default=None, *, required=False):
    value = os.environ.get(name) or default
    if required and not value:
        raise MissingConfiguration(
            f"Environment variable {name!r} is not set.\n"
            f"Copy .env.example to .env and fill it in, or export {name} before "
            f"starting the kernel. See README.md -> '.env configuration'."
        )
    return value


API_BASE = env("AML_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_TIMEOUT_S = float(env("AML_API_TIMEOUT_S", "180"))
EXPECTED_SEED_VERSION = env("AML_EXPECTED_SEED_VERSION", "scenarios-v1")
RESET_BEFORE_RUN = env("AML_RESET_BEFORE_RUN", "false").lower() in ("1", "true", "yes")

# Every notebook needs an OpenAI key. ToolCorrectnessMetric scores without any
# LLM call, but DeepEval 4.1.4 still builds a GPTModel in its constructor and
# raises without a key, so the key is required there too - just never used.
JUDGE_MODEL = env("DEEPEVAL_JUDGE_MODEL", "gpt-5.4-mini")
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# One key per role, falling back to the single AML_API_KEY. Blank is correct
# when the application runs with AUTH_MODE=off (its default).
API_KEYS = {
    "analyst": env("AML_API_KEY_ANALYST") or env("AML_API_KEY", ""),
    "eval_reader": env("AML_API_KEY_EVAL_READER") or env("AML_API_KEY", ""),
    "test_operator": env("AML_API_KEY_TEST_OPERATOR") or env("AML_API_KEY", ""),
}

print(f"API base URL       : {API_BASE}")
print(f"Request timeout    : {API_TIMEOUT_S}s")
print(f"Judge model        : {JUDGE_MODEL}")
print(f"Expected seed      : {EXPECTED_SEED_VERSION}")
print(f"API key configured : {bool(API_KEYS['analyst'])}  (False is correct when AUTH_MODE=off)")
print(f"OPENAI_API_KEY set : {bool(os.environ.get('OPENAI_API_KEY'))}")

In [ ]:
# --------------------------------------------------------------------------
# A small HTTP client. Every failure mode the application can present is
# turned into a message that names the cause and the thing to check.
# --------------------------------------------------------------------------
# The contract these notebooks were written against. The application may
# serve a HIGHER minor version: a MINOR bump is additive by its own
# contract policy (1.0.0 -> 1.1.0 added HealthResponse.build_version and
# changed nothing else), so treating it as a mismatch would turn this
# guard into noise on every single call. Only a MAJOR change, or an
# application older than these notebooks, is a problem.
EXPECTED_SCHEMA_VERSION = "1.0.0"


def contract_version(value):
    """(major, minor) from a MAJOR.MINOR.PATCH string, or None."""
    try:
        parts = value.split("+")[0].split(".")
        return int(parts[0]), int(parts[1])
    except (AttributeError, IndexError, ValueError):
        return None

SECRET_KEY_HINTS = ("api_key", "apikey", "authorization", "secret", "password",
                    "credential", "token")


def redact(value):
    """Mask credential-like values before anything is printed."""
    if isinstance(value, dict):
        return {
            k: ("***REDACTED***" if any(h in k.lower() for h in SECRET_KEY_HINTS)
                else redact(v))
            for k, v in value.items()
        }
    if isinstance(value, list):
        return [redact(v) for v in value]
    return value


class ApiError(RuntimeError):
    """A non-2xx response, carrying the application's error envelope."""


def api(method, path, *, role="analyst", json_body=None, params=None,
        expect_status=None):
    """Call the application API and return parsed JSON.

    role selects which API key is sent. It only matters when the application
    runs with AUTH_MODE=api_key; with AUTH_MODE=off the header is omitted.
    """
    headers = {"Accept": "application/json"}
    key = API_KEYS.get(role, "")
    if key:
        headers["X-API-Key"] = key

    url = f"{API_BASE}{path}"
    try:
        response = httpx.request(method, url, headers=headers, json=json_body,
                                 params=params, timeout=API_TIMEOUT_S)
    except httpx.ConnectError as exc:
        raise ApiError(
            f"Could not connect to {url}.\n"
            f"  - Is the application running?  curl {API_BASE}/api/health\n"
            f"  - Is AML_API_BASE_URL correct? It is currently {API_BASE!r}.\n"
            f"  - Underlying error: {exc}"
        ) from exc
    except httpx.TimeoutException as exc:
        raise ApiError(
            f"{method} {url} timed out after {API_TIMEOUT_S}s.\n"
            f"  - An investigation run does retrieval, several MCP tool calls and\n"
            f"    one LLM synthesis; raise AML_API_TIMEOUT_S if this is expected.\n"
            f"  - Underlying error: {exc!r}"
        ) from exc

    served = response.headers.get("X-Schema-Version")
    served_version = contract_version(served) if served else None
    expected_version = contract_version(EXPECTED_SCHEMA_VERSION)
    if served_version and served_version[0] != expected_version[0]:
        raise ApiError(
            f"The application serves contract version {served}; these notebooks were "
            f"written against {EXPECTED_SCHEMA_VERSION}. A MAJOR change means fields "
            f"may have been removed or retyped - re-derive the goldens against the "
            f"new contract rather than scoring against one they do not match."
        )
    if served_version and served_version[1] < expected_version[1]:
        print(f"WARNING: application reports contract version {served}, older than "
              f"the {EXPECTED_SCHEMA_VERSION} these notebooks were written against. "
              f"Fields the goldens rely on may not exist yet.")

    if response.status_code >= 400:
        try:
            envelope = response.json()
        except ValueError:
            envelope = {"raw_body": response.text[:1000]}
        hint = {
            401: "AUTH_MODE=api_key is on and no valid X-API-Key was sent. Set AML_API_KEY.",
            403: "The key's role may not reach this endpoint. eval_reader is needed for "
                 "/api/agent/trace and /api/eval/*; test_operator for /api/dev/reset and "
                 "/api/mcp/invoke.",
            404: "The id does not exist. Resolve ids from GET /api/eval/scenarios rather "
                 "than hardcoding them.",
            409: "Often index_not_built - the vector index has never been built. "
                 "POST /api/dev/reset once, or set AML_RESET_BEFORE_RUN=true.",
            502: "The application's LLM provider failed or returned output that broke its "
                 "own schema contract. Retry, or inspect GET /api/agent/trace/{run_id}.",
            503: "llm_not_configured - the application has no OPENROUTER_API_KEY. "
                 "This is the application's key, not the judge's OPENAI_API_KEY.",
        }.get(response.status_code, "")
        raise ApiError(
            f"{method} {url} -> HTTP {response.status_code}\n"
            f"  envelope: {json.dumps(envelope, indent=2)[:1200]}\n"
            + (f"  hint: {hint}" if hint else "")
        )

    if expect_status is not None and response.status_code != expect_status:
        raise ApiError(f"{method} {url} -> expected HTTP {expect_status}, "
                       f"got {response.status_code}")

    if not response.content:
        return None
    try:
        return response.json()
    except ValueError as exc:
        raise ApiError(
            f"{method} {url} returned HTTP {response.status_code} but the body is not "
            f"JSON.\n  first 500 bytes: {response.text[:500]!r}"
        ) from exc


def show(title, payload, limit=2500):
    """Pretty-print a payload with secrets masked and long bodies truncated."""
    text = json.dumps(redact(payload), indent=2, default=str)
    print(f"----- {title} -----")
    print(text if len(text) <= limit else text[:limit] + f"\n... [{len(text) - limit} more characters]")


health = api("GET", "/api/health")
show("GET /api/health", health)
if not health.get("status") == "ok":
    raise ApiError(f"Application is not healthy: {health}")

## Endpoints exercised

| Endpoint | Role here |
|---|---|
| `POST /api/cases/{case_id}/investigate` | The agent's MCP tool calls, in execution order |
| `POST /api/mcp/invoke` | Two further real MCP invocations |
| `GET /api/mcp/servers` | Server names and connection status |
| `GET /api/mcp/tools` | Each tool's `input_schema`, straight from the MCP protocol |
| `GET /api/eval/export/{run_id}` | The task statement and the final assessment |

The application is a genuinely good fit for the *tool* half of this metric: it runs three
real MCP servers over stdio and publishes each tool's own JSON Schema, so `available_tools`
on each declared server is the servers' own declaration rather than a copy.

It is a poor fit for the *multi-turn* half, for the reason set out below.

### How the conversation is assembled, and what that costs

The application has **no multi-turn tool-calling endpoint**. The two AI surfaces divide the
problem the other way round:

| Surface | Multi-turn? | Calls tools? |
|---|---|---|
| `POST /api/rag/query` | yes, case-scoped conversation memory | no - `tools_called` is always `[]` by design |
| `POST /api/cases/{id}/investigate` | no - single request, no conversation state | yes - full MCP tool calls |
| `POST /api/mcp/invoke` | no | one tool, invoked directly |

So a `ConversationalTestCase` carrying tool calls cannot be obtained from any single call.
This notebook builds one from **three real API calls against the same case**, and it is
important to be precise about which parts are the application's and which are the harness's:

**Real, straight from the API** - every tool name, every argument set, every tool result,
every status, the execution order within the investigation, and the final assessment text.

**Imposed by the harness** - the framing of those calls as a conversation: the user turns,
and the decision to present each tool call as its own assistant turn.

That framing is not arbitrary. `MultiTurnMCPUseMetric` groups turns into unit interactions
and **skips any unit of two turns or fewer**, so a flat user/assistant alternation yields no
tasks at all and scores `0.0` with an empty reason - a false failure. The metric expects the
shape a real MCP agent produces: one user message, then one assistant turn per tool call, then
the answer. Laying the calls out that way matches the execution order the trace records.

The honest summary: this scores the application's **tool-using behaviour** faithfully, and
its **conversational** behaviour not at all, because there is nothing conversational to
score. It is recorded as *Implemented with limitations* in `metric-notes.md`, with the
application change that would remove the caveat.

In [ ]:
# --------------------------------------------------------------------------
# Resolve scenarios to live row ids. Seed ids are assigned by insert order, so
# a hardcoded case_id silently rebinds to a different case when the seed data
# changes. GET /api/eval/scenarios exists precisely to avoid that.
# --------------------------------------------------------------------------
if RESET_BEFORE_RUN:
    # Drops and recreates every table, restoring deterministic seed state.
    reset = api("POST", "/api/dev/reset", role="test_operator")
    show("POST /api/dev/reset", reset)

SCENARIOS = {s["scenario_id"]: s for s in api("GET", "/api/eval/scenarios",
                                              role="eval_reader")}

seed_versions = {s["seed_version"] for s in SCENARIOS.values()}
if seed_versions != {EXPECTED_SEED_VERSION}:
    raise RuntimeError(
        f"Seed version mismatch: application reports {seed_versions}, the goldens in "
        f"this notebook were authored against {EXPECTED_SEED_VERSION!r}.\n"
        f"A golden authored against different seed data is not a weaker test, it is a "
        f"wrong one - fix the seed or the golden rather than lowering the threshold."
    )

for sid, s in sorted(SCENARIOS.items()):
    print(f"{sid}: case_id={s['case_id']} customer_id={s['customer_id']} "
          f"transaction_id={s['transaction_id']}  {s['title']}")

In [ ]:
# --------------------------------------------------------------------------
# The exact requests: one investigation, then two direct tool invocations on
# the same case.
# --------------------------------------------------------------------------
CASE_ID = SCENARIOS["s3"]["case_id"]   # "Possible sanctions name match (beneficiary near-miss)"

case = api("GET", f"/api/cases/{CASE_ID}")
beneficiary_country = None
if case.get("transaction") and case["transaction"].get("beneficiary_id"):
    beneficiary_country = api(
        "GET", f"/api/beneficiaries/{case['transaction']['beneficiary_id']}")["country"]

print("1.", "POST", f"{API_BASE}/api/cases/{CASE_ID}/investigate")
print("   body: (none - this endpoint reads no request body)")

DIRECT_PROBES = [
    {
        "user_intent": (f"Is {beneficiary_country} on the internal high-risk jurisdiction "
                        f"list?"),
        "request": {"server": "risk_screening", "tool": "check_high_risk_country",
                    "arguments": {"country_code": beneficiary_country}},
    },
    {
        "user_intent": "Find the sanctions screening extract filed on this case.",
        "request": {"server": "evidence", "tool": "search_case_documents",
                    "arguments": {"case_id": CASE_ID,
                                  "query": "sanctions screening extract"}},
    },
]
for n, probe in enumerate(DIRECT_PROBES, start=2):
    print(f"{n}.", "POST", f"{API_BASE}/api/mcp/invoke")
    print("   body:", json.dumps(probe["request"]))

In [ ]:
# --------------------------------------------------------------------------
# The raw responses.
# --------------------------------------------------------------------------
investigation = api("POST", f"/api/cases/{CASE_ID}/investigate", expect_status=201)
RUN_ID = investigation["run_id"]
export = api("GET", f"/api/eval/export/{RUN_ID}", role="eval_reader")

print(f"investigation run_id = {RUN_ID}")
print(f"tool calls made       = {len(investigation['tools_called'])}")
for call in investigation["tools_called"]:
    print(f"  {call['server']}.{call['tool']:<28} status={call['status']:<12} "
          f"args={json.dumps(call['input'])}")
    print(f"      result: {json.dumps(call['output'])[:160]}")
print()

direct_invocations = []
for probe in DIRECT_PROBES:
    response_body = api("POST", "/api/mcp/invoke", role="test_operator",
                        json_body=probe["request"])
    direct_invocations.append({**probe, "response": response_body})
    print(f"POST /api/mcp/invoke {response_body['namespaced_name']} -> "
          f"status={response_body['status']} latency_ms={response_body['latency_ms']}")
    print(f"  output: {json.dumps(response_body['output'])[:200]}")

failed = [c for c in investigation["tools_called"] if c["status"] != "success"]
if len(failed) == len(investigation["tools_called"]) and failed:
    raise RuntimeError(
        "Every tool call in this run failed, so there is no successful tool use to score. "
        "Check GET /api/mcp/servers - a server that is down reports status 'unavailable'."
    )

In [ ]:
# --------------------------------------------------------------------------
# The live tool catalogue: names, descriptions and JSON Schemas straight from
# the MCP protocol, so nothing here is a hand-maintained copy that can drift.
# --------------------------------------------------------------------------
servers_status = api("GET", "/api/mcp/servers", role="eval_reader")
print("MCP servers:")
for server in servers_status:
    print(f"  {server['name']:<16} status={server['status']:<12} "
          f"tools={server['tool_count']}  error={server['error']}")

disconnected = [s["name"] for s in servers_status if s["status"] != "connected"]
if disconnected:
    print(f"\nWARNING: {disconnected} are not connected. Their tools will be missing "
          f"from the catalogue, which changes what the judge considers available.")

mcp_tools = api("GET", "/api/mcp/tools", role="eval_reader")
print()
for server in mcp_tools:
    print(f"{server['server']}:")
    for tool in server["tools"]:
        required = tool["input_schema"].get("required", [])
        print(f"  {tool['namespaced_name']:<42} required={required}")

## Mapping the API response onto DeepEval fields

| DeepEval field | API source | Note |
|---|---|---|
| `turns[].mcp_tools_called[].name` | `f"{server}.{tool}"`, or `namespaced_name` from the invoke response | |
| `turns[].mcp_tools_called[].args` | `tools_called[].input` / the invoke request's `arguments` | Unmodified |
| `turns[].mcp_tools_called[].result` | `tools_called[].output` wrapped in `CallToolResult` | See below |
| `mcp_servers[].server_name` | `GET /api/mcp/servers` -> `name` | |
| `mcp_servers[].transport` | `"stdio"` | The application's MCP config declares stdio for all three servers |
| `mcp_servers[].available_tools` | `mcp.types.Tool` built from `namespaced_name`, `description`, `input_schema` | |

### Two representation details, stated plainly

**Re-wrapping the result.** The API returns each tool's output as a plain JSON object,
because the orchestrator has already unwrapped the MCP envelope. DeepEval requires an
`mcp.types.CallToolResult`, so the payload is put back into that envelope: the JSON as
`TextContent`, the object as `structuredContent`, and `isError` derived from the API's own
`status` field. No value is changed - this restores the wire shape the payload arrived in.

**The `"result"` nesting.** DeepEval 4.1.4 reads `result.structuredContent["result"]`, so
the payload is nested under a `result` key. This is a presentation requirement of the
metric, not a change to the data.

**A hard version constraint.** `mcp` 1.28.0 renamed `CallToolResult.structuredContent` to
`structured_content`, while deepeval 4.1.4 still reads the camelCase attribute. With
`mcp >= 1.28` this metric raises `AttributeError`. `pyproject.toml` therefore pins
`mcp>=1.12,<1.28`, and the reason is recorded in `metric-notes.md`.

In [ ]:
# --------------------------------------------------------------------------
# Assemble the conversation from the three real calls.
#
# mcp.types.CallToolResult is required by DeepEval's Turn validator, so each
# API tool result is re-wrapped into the MCP wire shape it originally had:
# the JSON payload as text content, plus structuredContent, plus isError
# derived from the API's own status field. No value is altered.
#
# One presentation detail: DeepEval 4.1.4 reads
# result.structuredContent["result"], so the payload is nested under a
# "result" key to match what it expects.
# --------------------------------------------------------------------------
from deepeval.test_case import ConversationalTestCase, Turn, ToolCall
from deepeval.test_case.mcp import MCPToolCall
from mcp.types import CallToolResult, TextContent


def as_call_tool_result(output, status):
    return CallToolResult(
        content=[TextContent(type="text", text=json.dumps(output))],
        structuredContent={"result": output},
        isError=status != "success",
    )


turns = [Turn(role="user", content=export["input"])]

# One assistant turn per MCP tool call, in the order the agent made them.
for call in investigation["tools_called"]:
    name = f"{call['server']}.{call['tool']}"
    turns.append(Turn(
        role="assistant",
        content=f"Calling MCP tool {name} with {json.dumps(call['input'])}.",
        tools_called=[ToolCall(name=name, input_parameters=call["input"],
                               output=call["output"])],
        mcp_tools_called=[MCPToolCall(name=name, args=call["input"],
                                      result=as_call_tool_result(call["output"],
                                                                 call["status"]))],
    ))

# The agent's final assessment closes the first unit interaction.
turns.append(Turn(role="assistant", content=export["actual_output"]))

# Two further exchanges from real POST /api/mcp/invoke calls.
for probe in direct_invocations:
    request_body, response_body = probe["request"], probe["response"]
    turns.append(Turn(role="user", content=probe["user_intent"]))
    turns.append(Turn(
        role="assistant",
        content=(f"Calling MCP tool {response_body['namespaced_name']} with "
                 f"{json.dumps(request_body.get('arguments', {}))}."),
        tools_called=[ToolCall(name=response_body["namespaced_name"],
                               input_parameters=request_body.get("arguments", {}),
                               output=response_body["output"])],
        mcp_tools_called=[MCPToolCall(
            name=response_body["namespaced_name"],
            args=request_body.get("arguments", {}),
            result=as_call_tool_result(response_body["output"],
                                       response_body["status"]))],
    ))
    turns.append(Turn(role="assistant",
                      content=json.dumps(response_body["output"])[:1200]))

print(f"assembled {len(turns)} turns")
for n, turn in enumerate(turns):
    tools = [t.name for t in (turn.tools_called or [])]
    print(f"  [{n}] {turn.role:<9} {'tools=' + str(tools) if tools else '':<52} "
          f"{turn.content[:60].replace(chr(10), ' ')}...")

In [ ]:
# --------------------------------------------------------------------------
# Declare the MCP servers, then build the test case.
#
# available_tools are mcp.types.Tool objects constructed from the schemas the
# servers themselves report, so the declaration cannot drift from reality.
# --------------------------------------------------------------------------
from deepeval.test_case.mcp import MCPServer
from mcp.types import Tool as MCPTool

status_by_name = {s["name"]: s for s in servers_status}

mcp_servers = []
for server in mcp_tools:
    mcp_servers.append(MCPServer(
        server_name=server["server"],
        transport="stdio",          # as declared in the application's MCP config
        available_tools=[
            MCPTool(name=tool["namespaced_name"],
                    description=tool["description"],
                    inputSchema=tool["input_schema"])
            for tool in server["tools"]
        ],
    ))

test_case = ConversationalTestCase(turns=turns, mcp_servers=mcp_servers)

print(f"MCP SERVERS DECLARED: {len(test_case.mcp_servers)}")
for server in test_case.mcp_servers:
    connected = status_by_name.get(server.server_name, {}).get("status")
    print(f"  {server.server_name:<16} transport={server.transport:<10} "
          f"status={connected} tools={len(server.available_tools)}")
    for tool in server.available_tools:
        print(f"      {tool.name:<44} required={tool.inputSchema.get('required', [])}")
print()
print("ACTUAL MCP TOOL CALLS, per turn, with ACTUAL ARGUMENTS")
for n, turn in enumerate(test_case.turns):
    for call in turn.mcp_tools_called or []:
        print(f"  turn {n}: {call.name:<42} args={json.dumps(call.args)}")
        print(f"           isError={call.result.isError}")
print()
print("EXPECTED TOOLS / EXPECTED PLAN : not used by this metric")

In [ ]:
# --------------------------------------------------------------------------
# Guard: the metric skips any unit interaction of two turns or fewer, and
# scores 0.0 with an empty reason when no unit survives. That is a false
# failure, and it looks exactly like a real one. Check the shape first.
# --------------------------------------------------------------------------
units, current, seen_user = [], [], False
for turn in test_case.turns:
    if current and current[-1].role == "assistant" and turn.role == "user" and seen_user:
        units.append(current)
        current, seen_user = [turn], True
        continue
    current.append(turn)
    seen_user = seen_user or turn.role == "user"
if current and len(current) > 1 and current[-1].role == "assistant" and seen_user:
    units.append(current)

scorable = [u for u in units if len(u) > 2]
print(f"unit interactions        : {len(units)}  sizes={[len(u) for u in units]}")
print(f"scorable (more than 2)   : {len(scorable)}")

if not scorable:
    raise RuntimeError(
        "No unit interaction has more than two turns, so MultiTurnMCPUseMetric would "
        "extract no tasks and return 0.0 with an empty reason - a false failure.\n"
        "Each user turn must be followed by more than one assistant turn (one per tool "
        "call, then the answer)."
    )

## Judge and threshold

- **Judge model**: `DEEPEVAL_JUDGE_MODEL`, default `gpt-5.4-mini`.
- **Threshold**: `0.5`, DeepEval's documented default for `MultiTurnMCPUseMetric`.

The default is kept. Because the final score is `min(primitive_accuracy, argument_accuracy)`
it is bounded by whichever half is weaker, which makes a pass at `0.5` mean "neither
protocol-level dimension collapsed" rather than "tool use was good".

In authoring runs this scored around `0.82`, with argument accuracy at `1.0` across every
interaction (unsurprising - the arguments are constructed by a rule engine against schemas
it was written for) and primitive accuracy pulled down by the judge noting a document search
that could have been broader. That split is the interesting output of this metric, and the
result cell prints both sub-score lists for exactly that reason.

In [ ]:
from deepeval.metrics import MultiTurnMCPUseMetric

metric = MultiTurnMCPUseMetric(
    threshold=0.5,          # DeepEval's documented default
    model=JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
    verbose_mode=True,
)
print(f"metric class : {type(metric).__name__}")
print(f"judge model  : {JUDGE_MODEL}")
print(f"threshold    : {metric.threshold}")
print(f"async_mode   : {metric.async_mode}")
print(f"strict_mode  : {metric.strict_mode}")

In [ ]:
# --------------------------------------------------------------------------
# Run the metric. A judge failure is caught and explained rather than left as
# a bare traceback, because "the judge could not be reached" and "the
# application scored badly" are completely different findings.
# --------------------------------------------------------------------------
try:
    metric.measure(test_case)
except Exception as exc:                      # noqa: BLE001 - diagnostic wrapper
    message = str(exc)
    print(f"METRIC EXECUTION FAILED: {type(exc).__name__}: {message[:600]}")
    if "api_key" in message.lower() or "authentication" in message.lower():
        print("  -> OPENAI_API_KEY is missing or rejected. This is the judge's key, "
              "not the application's.")
    elif "model" in message.lower() and "not" in message.lower():
        print(f"  -> The judge model {JUDGE_MODEL!r} was rejected. Check that your "
              f"OpenAI account can reach it, and that the installed DeepEval version "
              f"knows the id. Set DEEPEVAL_JUDGE_MODEL to change it.")
    elif "rate" in message.lower():
        print("  -> Rate limited by the judge provider. Re-run the cell.")
    raise

In [ ]:
# --------------------------------------------------------------------------
# Score, verdict, reason and debug output.
#
# Read `metric.is_successful()`, never the raw score: DeepEval metrics do not
# all point the same way. AnswerRelevancy and ToolCorrectness are "higher is
# better"; Bias and Hallucination are rates where lower is better; PIILeakage
# is a privacy score where 0.0 means maximum leakage. is_successful() applies
# the correct comparison for the metric.
# --------------------------------------------------------------------------
print(f"metric          : {type(metric).__name__}")
print(f"judge model     : {JUDGE_MODEL}")
print(f"threshold       : {metric.threshold}")
print(f"score           : {metric.score}")
print(f"PASS / FAIL     : {'PASS' if metric.is_successful() else 'FAIL'}")
print(f"judge cost (USD): {metric.evaluation_cost}")
print()
print("reason:")
print(textwrap.fill(str(metric.reason), width=96, subsequent_indent="  "))
print()
print("----- verbose judge log (debug) -----")
print(metric.verbose_logs or "(none - construct the metric with verbose_mode=True)")

In [ ]:
# --------------------------------------------------------------------------
# The two sub-scores. The combined score is min(mean(tool), mean(args)), so
# knowing which half bound it is the whole diagnostic value of this metric.
# --------------------------------------------------------------------------
print("PRIMITIVE (TOOL) ACCURACY, per unit interaction")
for n, (score, reason) in enumerate(getattr(metric, "tools_scores_reasons_list", []) or [],
                                    start=1):
    print(f"  [{n}] score={score}")
    print(textwrap.fill(str(reason), width=92, initial_indent="      ",
                        subsequent_indent="      "))
print()
print("ARGUMENT ACCURACY, per unit interaction")
for n, (score, reason) in enumerate(getattr(metric, "args_scores_reasons_list", []) or [],
                                    start=1):
    print(f"  [{n}] score={score}")
    print(textwrap.fill(str(reason), width=92, initial_indent="      ",
                        subsequent_indent="      "))

## Limitations in a black-box acceptance test

1. **The multi-turn dimension is not really being tested.** The application exposes no
   conversational MCP agent, so the transcript is assembled from single-turn calls. The
   MCP-usage findings are sound; any claim about behaviour *across* turns is not.
2. **Only tools are exercised.** MCP also defines resources and prompts, and the metric
   scores `mcp_resources_called` and `mcp_prompts_called` when present. This application
   exposes neither through its API, so those dimensions are untested and no amount of
   harness work can change that.
3. **The result envelope is reconstructed.** `CallToolResult` is rebuilt from the API's
   unwrapped payload. Content type, annotations and any `_meta` the servers originally sent
   are lost before the harness ever sees them - the orchestrator has already discarded
   them. A metric that reasoned about content blocks would be scoring the reconstruction.
4. **A narrow dependency pin is load-bearing.** `mcp>=1.12,<1.28` is required for
   compatibility with deepeval 4.1.4. Anyone loosening that bound in a copied project will
   see this notebook fail with an `AttributeError`, not a graceful message.
5. **Argument accuracy is close to guaranteed here.** Arguments are built by a rule engine
   against the same schemas the metric validates them with, so a perfect argument sub-score
   confirms the rules were written correctly rather than that an agent reasoned well. The
   metric would earn its keep against an LLM-driven tool caller.